# SpaceX Interactive Folium Launch Map

In [8]:

!pip -q install folium

import pandas as pd
import folium
from folium.plugins import MarkerCluster
from math import radians, sin, cos, asin, sqrt
from google.colab import files

df = pd.read_csv("dataset_part_2.csv")

print("Total launches:", len(df))
print("Total launch sites:", df["LaunchSite"].nunique())
print("Launch sites:", df["LaunchSite"].unique())

m = folium.Map(
    location=[32, -100],
    zoom_start=4,
    tiles="Esri WorldImagery",
    attr="Esri",
    control_scale=True
)

cluster = MarkerCluster(
    name="SpaceX Launch Sites"
).add_to(m)

for site, g in df.groupby("LaunchSite"):
    lat = float(g["Latitude"].iloc[0])
    lon = float(g["Longitude"].iloc[0])

    folium.Marker(
        location=[lat, lon],
        tooltip=site,
        popup=folium.Popup(
            f"<b>{site}</b><br>"
            f"Total Launches: {len(g)}<br>"
            f"Success Rate: {g['Class'].mean():.1%}",
            max_width=300
        ),
        icon=folium.Icon(
            color="blue",
            icon="info-sign"
        )
    ).add_to(cluster)

    folium.Circle(
        location=[lat, lon],
        radius=1000,
        color="yellow",
        weight=2,
        fill=False
    ).add_to(m)

outcomes = folium.FeatureGroup(
    name="Landing Outcomes",
    show=False
)

for _, r in df.iterrows():
    folium.CircleMarker(
        location=[
            float(r["Latitude"]),
            float(r["Longitude"])
        ],
        radius=4,
        color="green" if r["Class"] == 1 else "red",
        fill=True,
        fill_color="green" if r["Class"] == 1 else "red",
        fill_opacity=0.5,
        tooltip=(
            "Successful Landing"
            if r["Class"] == 1
            else "Unsuccessful Landing"
        )
    ).add_to(outcomes)

outcomes.add_to(m)

folium.LayerControl().add_to(m)

site_coords = df.groupby("LaunchSite")[
    ["Latitude", "Longitude"]
].first()

print("\nLaunch Site Coordinates:")
display(site_coords)

def haversine(a, b, c, d):
    R = 6371
    da = radians(c - a)
    db = radians(d - b)

    h = (
        sin(da / 2)**2
        + cos(radians(a))
        * cos(radians(c))
        * sin(db / 2)**2
    )

    return 2 * R * asin(sqrt(h))

print("\nDistances Between Launch Sites:")

for i, a in enumerate(site_coords.index):
    for b in site_coords.index[i + 1:]:
        distance = haversine(
            *site_coords.loc[a],
            *site_coords.loc[b]
        )

        print(
            a, "-", b,
            round(distance, 1), "km"
        )

m.save("launch_sites_folium.html")

display(m)

files.download("launch_sites_folium.html")


Total launches: 90
Total launch sites: 3
Launch sites: ['CCAFS SLC 40' 'VAFB SLC 4E' 'KSC LC 39A']

Launch Site Coordinates:


,Latitude,Longitude
LaunchSite,,
CCAFS SLC 40,28.561857,-80.577366
KSC LC 39A,28.608058,-80.603956
VAFB SLC 4E,34.632093,-120.610829



Distances Between Launch Sites:
CCAFS SLC 40 - KSC LC 39A 5.8 km
CCAFS SLC 40 - VAFB SLC 4E 3825.9 km
KSC LC 39A - VAFB SLC 4E 3821.6 km


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>